In [0]:
from pyspark.sql import functions as F

silver = spark.table("silver_order_details")

# 1. Monthly Revenue Analysis
monthly_revenue = (
    silver
    .withColumn("year_month", F.date_format("purchase_timestamp", "yyyy-MM"))
    .groupBy("year_month")
    .agg(F.round(F.sum("payment_value"), 2).alias("total_revenue"))
    .orderBy("year_month")
)
monthly_revenue.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_monthly_revenue")
monthly_revenue.show()

# 2. Top Selling Product Categories
top_categories = (
    silver
    .groupBy("product_category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.round(F.sum("price"), 2).alias("total_revenue"),
    )
    .orderBy(F.desc("total_revenue"))
)
top_categories.write.format("delta").mode("overwrite").saveAsTable("gold_top_categories")
top_categories.show(10)

# 3. Payment Method Analysis
payment_analysis = (
    silver
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("total_transactions"),
        F.round(F.sum("payment_value"), 2).alias("revenue_contribution"),
    )
    .orderBy(F.desc("revenue_contribution"))
)
payment_analysis.write.format("delta").mode("overwrite").saveAsTable("gold_payment_analysis")
payment_analysis.show()

# 4. Customer City Analysis
city_analysis = (
    silver
    .groupBy("customer_city")
    .agg(
        F.countDistinct("order_id").alias("number_of_orders"),
        F.round(F.sum("payment_value"), 2).alias("revenue"),
    )
    .orderBy(F.desc("revenue"))
)
city_analysis.write.format("delta").mode("overwrite").saveAsTable("gold_city_analysis")
city_analysis.show(10)

print("Done! Tables created: gold_monthly_revenue, gold_top_categories, gold_payment_analysis, gold_city_analysis")

+----------+-------------+
|year_month|total_revenue|
+----------+-------------+
|   2016-09|       347.52|
|   2016-10|     61188.62|
|   2016-12|        19.62|
|   2017-01|    144553.34|
|   2017-02|    300460.77|
|   2017-03|    452894.25|
|   2017-04|    428330.73|
|   2017-05|    618831.53|
|   2017-06|    536273.53|
|   2017-07|    627487.75|
|   2017-08|    707114.89|
|   2017-09|    755097.28|
|   2017-10|     821099.6|
|   2017-11|   1269735.96|
|   2017-12|    908692.75|
|   2018-01|   1179189.77|
|   2018-02|   1029491.93|
|   2018-03|   1210762.79|
|   2018-04|    1239270.7|
|   2018-05|   1238243.15|
+----------+-------------+
only showing top 20 rows
+--------------------+------------+-------------+
|    product_category|total_orders|total_revenue|
+--------------------+------------+-------------+
|        beleza_saude|        8835|   1241285.95|
|  relogios_presentes|        5624|   1232603.14|
|     cama_mesa_banho|        9417|   1010948.19|
|       esporte_lazer|     

In [0]:
display(spark.table("gold_monthly_revenue"))

year_month,total_revenue
2016-09,347.52
2016-10,61188.62
2016-12,19.62
2017-01,144553.34
2017-02,300460.77
2017-03,452894.25
2017-04,428330.73
2017-05,618831.53
2017-06,536273.53
2017-07,627487.75


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.